# 05. Vehicle Plate Detection & OCR Pipeline

## 1. Overview
- **Tasks**: License Plate Detection & Optical Character Recognition (OCR)
- **Tools**:
  1. **Plate Detector**: RF-DETR (`Rikkosse/rfdetr_licences_plate_detector` / Roboflow RF-DETR Large) [Apache-2.0]
  2. **Plate OCR**: PaddleOCR / PaddleX [Apache-2.0]
- **Trigger Architecture**: Triggered conditionally only on vehicles (cars, trucks, motorcycles) from gate/entryway cameras to conserve GPU cycles.

### Pipeline Steps
1. **Vehicle Crop Ingestion**: The system crops the vehicle detected by RT-DETR.
2. **Plate Localization**: RF-DETR outputs normalized `[cx, cy, w, h]` for the plate.
3. **Image Enhancement**: Grayscale, CLAHE (Contrast Limited Adaptive Histogram Equalization), and noise filtering.
4. **Text Recognition**: PaddleOCR extracts Thai / Latin alphanumeric characters.
5. **Multi-Frame Majority Voting**: Consolidates readings over several frames to eliminate motion blur artifacts.

In [ ]:
# Import dependencies
import os
import cv2
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
from collections import Counter

## 2. RF-DETR License Plate Detector

In [ ]:
class PlateDetectorRFDetr:
    """RF-DETR ONNX model for plate detection on vehicle crops."""

    INPUT_SIZE = 768
    _MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
    _STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    def __init__(self, model_path: str = None, conf_threshold: float = 0.30):
        self.conf = conf_threshold
        self.session = None
        if model_path and os.path.exists(model_path):
            providers = ['MIGraphXExecutionProvider', 'CPUExecutionProvider']
            self.session = ort.InferenceSession(model_path, providers=providers)
            self.input_name = self.session.get_inputs()[0].name
        else:
            print('RF-DETR model not found. Running in simulation mode.')

    def detect_plate(self, vehicle_bgr_crop: np.ndarray):
        h0, w0 = vehicle_bgr_crop.shape[:2]
        if self.session is None:
            # Mock detection: center-bottom of vehicle crop
            x1, y1 = int(w0 * 0.35), int(h0 * 0.70)
            x2, y2 = int(w0 * 0.65), int(h0 * 0.85)
            return [(x1, y1, x2, y2, 0.94)]

        # Preprocess to 768x768 RGB
        rgb = cv2.cvtColor(vehicle_bgr_crop, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (self.INPUT_SIZE, self.INPUT_SIZE))
        blob = (resized.astype(np.float32) / 255.0).transpose(2, 0, 1)[None]
        blob = (blob - self._MEAN) / self._STD

        outs = self.session.run(None, {self.input_name: blob})
        dets = outs[0][0]      # [300, 4] -> cx, cy, w, h
        labels = outs[1][0]    # [300, 2] -> logits
        scores = 1.0 / (1.0 + np.exp(-labels[:, 1]))

        keep = scores >= self.conf
        dets, scores = dets[keep], scores[keep]
        results = []
        for (cx, cy, bw, bh), score in zip(dets, scores):
            x1 = max(0, int((cx - bw / 2) * w0))
            y1 = max(0, int((cy - bh / 2) * h0))
            x2 = min(w0, int((cx + bw / 2) * w0))
            y2 = min(h0, int((cy + bh / 2) * h0))
            results.append((x1, y1, x2, y2, float(score)))
        return results

## 3. Plate Preprocessing & PaddleOCR Recognition

In [ ]:
def preprocess_plate_image(plate_crop: np.ndarray) -> np.ndarray:
    """Enhances contrast and clarifies plate characters."""
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    # Apply CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return enhanced

class PlateOCRModule:
    """PaddleOCR inference wrapper for license plate reading."""

    def __init__(self):
        self.ocr = None
        try:
            from paddleocr import PaddleOCR
            # Lang supports 'th' (Thai) or 'en'
            self.ocr = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)
        except Exception:
            print('PaddleOCR library not loaded or initialized. Using fallback simulator.')

    def read(self, plate_crop: np.ndarray):
        if self.ocr is None:
            # Simulation return for standard Thai car plate
            return [{'text': '1AB-4567', 'confidence': 0.95}]

        enhanced = preprocess_plate_image(plate_crop)
        results = self.ocr.ocr(enhanced, cls=True)
        reads = []
        if results and results[0]:
            for line in results[0]:
                txt, conf = line[1][0], line[1][1]
                reads.append({'text': txt, 'confidence': float(conf)})
        return reads

## 4. Multi-Frame Consensus Voting (`PlateTracker`)

In [ ]:
class PlateVotingConsensus:
    """Aggregates OCR results over time to produce high-confidence consensus."""

    def __init__(self):
        self.buffer = []

    def add_reading(self, plate_text: str, confidence: float):
        if confidence > 0.6:
            self.buffer.append((plate_text, confidence))

    def resolve(self):
        if not self.buffer:
            return None
        # Weighted vote
        weighted_scores = {}
        for text, conf in self.buffer:
            weighted_scores[text] = weighted_scores.get(text, 0.0) + conf
        best_plate = max(weighted_scores, key=weighted_scores.get)
        return best_plate, weighted_scores[best_plate] / len(self.buffer)

## 5. Pipeline Demonstration

In [ ]:
# Initialize components
detector = PlateDetectorRFDetr()
ocr_engine = PlateOCRModule()
voting = PlateVotingConsensus()

# Create synthetic car crop (400x500)
vehicle_crop = np.ones((400, 500, 3), dtype=np.uint8) * 80
cv2.rectangle(vehicle_crop, (150, 270), (350, 330), (220, 220, 220), -1)
cv2.putText(vehicle_crop, '1AB-4567', (160, 315), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3)

# Step 1: Detect plate location
plates = detector.detect_plate(vehicle_crop)
for x1, y1, x2, y2, score in plates:
    plate_patch = vehicle_crop[y1:y2, x1:x2]
    print(f"Detected plate box: [{x1}, {y1}, {x2}, {y2}] with score {score:.2f}")

    # Step 2: OCR on plate crop
    ocr_res = ocr_engine.read(plate_patch)
    for res in ocr_res:
        voting.add_reading(res['text'], res['confidence'])

# Step 3: Resolve consensus
final_plate, avg_conf = voting.resolve()
print(f"\nConsensus Plate Output: '{final_plate}' (Confidence: {avg_conf:.2f})")